In [1]:
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC

In [2]:
df = pd.read_csv('voice (1).csv')

#('male' / 'female' -> 1 / 0)
le = LabelEncoder()
df['label'] = le.fit_transform(df['label'])

X = df.drop('label', axis=1)
y = df['label']

In [3]:
splits = {'70:30': 0.30, '80:20': 0.20}
kernels = ['linear', 'poly', 'rbf']

In [ ]:
performance_records = []

for split_name, test_ratio in splits.items():
  # Split
  X_train, X_test, y_train, y_test = train_test_split(
      X, y, test_size=test_ratio, random_state=42
  )

  # Standardize features
  scaler = StandardScaler()
  X_train_scaled = scaler.fit_transform(X_train)
  X_test_scaled = scaler.transform(X_test)

  for kernel in kernels:
    svm_model = SVC(kernel=kernel)
    svm_model.fit(X_train_scaled, y_train)

    # accuracy
    y_pred = svm_model.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)

    performance_records.append({
        'Data Split': split_name,
        'Kernel': kernel.upper() if kernel != 'poly' else 'Polynomial',
        'Test Samples': len(y_test),
        'Accuracy': round(acc, 4),
        'Accuracy (%)': f'{acc * 100:.2f}%',
    })

In [5]:
results_table = pd.DataFrame(performance_records)
print(results_table.to_string(index=False))

Data Split     Kernel  Test Samples  Accuracy Accuracy (%)
     70:30     LINEAR           951    0.9706       97.06%
     70:30 Polynomial           951    0.9579       95.79%
     70:30        RBF           951    0.9811       98.11%
     80:20     LINEAR           634    0.9763       97.63%
     80:20 Polynomial           634    0.9716       97.16%
     80:20        RBF           634    0.9826       98.26%


Across both split ratios, the RBF kernel delivers the highest classification accuracy (98.11% on 70:30 and 98.26% on 80:20). The Linear kernel performs good (97.06% and 97.63%), confirming that key acoustic variables such as meanfun and IQR provide strong linear separability. Increasing the training portion from 70% (2,217 samples) to 80% (2,534 samples) improves test accuracy across all three kernels